# Cloud Run
## load-311nyc-doe-dohmh-data main.py

In [ ]:
import os
import json
from datetime import datetime
from google.cloud import bigquery
from google.cloud.bigquery import SchemaField
from sodapy import Socrata
import time

# --- Configuration Constants ---
SOCRATA_HOST = 'data.cityofnewyork.us'
SOCRATA_DATASET_ID = 'erm2-nwe9' # The resource ID for NYC 311 Service Requests
# WARNING: Temporarily reduced for manual testing to avoid 'request too large' error due to log volume.
# For production via Cloud Scheduler, you may decide to increase or decrease this value.
CHUNK_SIZE = 5000
ORDER_BY_FIELD = 'created_date' # Field to order by for initial load consistency

# Define the custom Socrata Query Language (SoQL) $where clause here.
# Example 1 (DOT only): SOCRATA_WHERE_CLAUSE = "agency = 'DOT'"
# Example 2 (Complex filter): SOCRATA_WHERE_CLAUSE = "agency = 'DOT' AND complaint_type = 'Street Light Out'"
# Set to None or empty string ('') to ingest all data without filtering.
SOCRATA_WHERE_CLAUSE = "(agency = 'DOE' OR agency = 'DOHMH') AND created_date >= '2021-01-01T00:00:00.000'"

# SECRET: Read the Socrata App Token from the Cloud Function environment variables
# Note: This is required for reliable API access.
SOCRATA_APP_TOKEN = os.environ.get('GP5yvTTPsMKDFu2HFlQ4WNcXg')

# BigQuery Configuration
# CRITICAL: You MUST manually replace 'YOUR_PROJECT_ID_HERE' with your actual
# Google Cloud Project ID where BigQuery is hosted.
BQ_PROJECT_ID = 'preston-cis-9440-s26'
BQ_DATASET_ID = 'final_project_raw_sr'
BQ_TABLE_ID = 'source_service_request'

# Initialize BigQuery Client
# When BQ_PROJECT_ID is specified, the client is initialized for that project.
bq_client = bigquery.Client(project=BQ_PROJECT_ID)


# Define the CORE BigQuery schema explicitly.
# This list is used for INITIAL table creation and for type enforcement (TIMESTAMP, FLOAT, STRING for incident_zip).
CORE_BQ_SCHEMA = [
    # General Identifiers and Dates (MUST be present for initial table)
    SchemaField("unique_key", "STRING", mode="REQUIRED", description="Unique ID for the service request"),
    SchemaField("created_date", "TIMESTAMP", description="Date and time the request was created"),
    SchemaField("closed_date", "TIMESTAMP", description="Date and time the request was closed"),
    # Agency and Complaint Details
    SchemaField("agency", "STRING", description="Agency code"),
    SchemaField("agency_name", "STRING", description="Full name of the agency"),
    SchemaField("complaint_type", "STRING", description="Type of complaint"),
    SchemaField("descriptor", "STRING", description="Further description of the complaint"),
    # Location Details
    SchemaField("location_type", "STRING", description="Location where the incident occurred"),
    SchemaField("incident_zip", "STRING", description="Zip code of the incident location"), # Explicitly set to STRING
    SchemaField("incident_address", "STRING", description="Street address of the incident"),
    # Geospatial data
    SchemaField("latitude", "FLOAT", description="Latitude coordinate"),
    SchemaField("longitude", "FLOAT", description="Longitude coordinate"),
    # Additional fields added for final project
    SchemaField("descriptor_2", "STRING", description="Additional details about the problem (formerly Complaint Type)"),
    SchemaField("street_name", "STRING", description="Street name of incident address provided by submitter"),
    SchemaField("cross_street_1", "STRING", description="First Cross street based on the geo validated incident location"),
    SchemaField("cross_street_2", "STRING", description="Second Cross street based on the geo validated incident location"),
    SchemaField("intersection_street_1", "STRING", description="First intersecting street based on geo validated incident location"),
    SchemaField("intersection_street_2", "STRING", description="Second intersecting street based on geo validated incident location"),
    SchemaField("address_type", "STRING", description="Type of incident location information available"),
    SchemaField("city", "STRING", description="City of the incident location provided by geovalidation"),
    SchemaField("landmark", "STRING", description="Landmark name if incident location is identified as a Landmark"),
    SchemaField("facility_type", "STRING", description="Type of city facility associated to the SR"),
    SchemaField("status", "STRING", description="Status of SR submitted"),
    SchemaField("due_date", "TIMESTAMP", description="Date when responding agency is expected to update the SR"),
    SchemaField("resolution_description", "STRING", description="Describes the last action taken on the SR by the responding agency"),
    SchemaField("resolution_action_updated_date", "TIMESTAMP", description="Date when responding agency last updated the SR"),
    SchemaField("community_board", "STRING", description="Community Board provided by geovalidation"),
    SchemaField("council_district", "STRING", description="City Council district where the service request is located"),
    SchemaField("police_precinct", "STRING", description="NYPD precinct where the service request is located"),
    SchemaField("bbl", "STRING", description="Borough Block and Lot, provided by geovalidation"),
    SchemaField("borough", "STRING", description="Borough provided by the submitter and confirmed by geovalidation"),
    SchemaField("x_coordinate_state_plane", "FLOAT", description="Geo validated, X coordinate of the incident location"),
    SchemaField("y_coordinate_state_plane", "FLOAT", description="Geo validated, Y coordinate of the incident location"),
    SchemaField("open_data_channel_type", "STRING", description="How the SR was submitted to 311"),
    SchemaField("park_facility_name", "STRING", description="Name of the Parks Dept facility if incident location is a Parks Dept facility"),
    SchemaField("park_borough", "STRING", description="Borough of incident if it is a Parks Dept facility"),
    SchemaField("vehicle_type", "STRING", description="Type of TLC vehicle if incident is a taxi"),
    SchemaField("taxi_company_borough", "STRING", description="Borough of the taxi company if incident is a taxi"),
    SchemaField("taxi_pick_up_location", "STRING", description="Taxi pick up location if incident is a taxi"),
    SchemaField("bridge_highway_name", "STRING", description="Bridge/Highway name if incident is identified as such"),
    SchemaField("bridge_highway_direction", "STRING", description="Direction where the issue took place on the Bridge/Highway"),
    SchemaField("road_ramp", "STRING", description="Differentiates if the issue was on the Road or the Ramp"),
    SchemaField("bridge_highway_segment", "STRING", description="Additional information on the section of the Bridge/Highway"),
# If you want to store the geo point as a geography type:
# SchemaField("location", "GEOGRAPHY", description="Geo point of the incident location"),
]

def clean_record(record):
    """
    Cleans up type anomalies (e.g., empty strings to None), removes Socrata
    internal metadata fields, and cleans field names for BigQuery compatibility.
    """
    cleaned_record = {}

    for key, value in record.items():
        # 1. Drop Socrata internal/metadata and computed region fields
        if key.startswith(':') or key in ['_id', '_submission_details', 'location'] or '__computed_region_' in key:
            continue

        # 2. Clean field names for BigQuery (replace problematic characters AND force lowercase)
        # Force lower() before replacing characters to ensure consistency.
        clean_key = key.lower().replace(':', '_').replace('@', '_').replace(' ', '_')

        # 3. Type normalization: Convert empty strings to None for proper NULL handling in BQ
        if value == "":
            cleaned_record[clean_key] = None
        else:
            cleaned_record[clean_key] = value

        # 4. Special handling for zip codes (FORCES STRING CONVERSION)
        # Ensures leading zeros are preserved and BigQuery receives a string.
        if 'zip' in clean_key.lower() and cleaned_record.get(clean_key) is not None:
             cleaned_record[clean_key] = str(cleaned_record[clean_key])

    return cleaned_record

def get_current_offset():
    """
    Reads the current row count from the BigQuery table. This count serves as the
    starting offset for the Socrata API call.
    """
    # Note: We must explicitly use the BQ_PROJECT_ID here since we are querying it.
    query = f"""
        SELECT COUNT(*) AS current_row_count
        FROM `{BQ_PROJECT_ID}.{BQ_DATASET_ID}.{BQ_TABLE_ID}`
    """
    print(f"Executing BQ query to find current row count (offset).")

    try:
        query_job = bq_client.query(query)
        results = query_job.result()

        for row in results:
            # The count will be 0 if the table is empty
            offset = row.current_row_count
            print(f"Retrieved current offset (row count) from BQ: {offset}")
            return offset

    except Exception as e:
        # If the table doesn't exist yet, the query will fail. We treat this as offset 0.
        print(f"Warning: Failed to retrieve count from BQ (likely table/dataset doesn't exist yet): {e}")

    # Default offset to 0 if BQ query fails (triggers full historical load)
    default_offset = 0
    print(f"Defaulting offset to {default_offset}. This will load historical data.")
    return default_offset

def update_bq_schema_if_needed(table_ref, data):
    """
    Checks the incoming data chunk against the current BigQuery table schema
    and adds any new fields found in the data to the table schema.
    """
    try:
        # Get the current table object and schema
        table = bq_client.get_table(table_ref)
        current_schema_fields = {field.name for field in table.schema}

        # Get all unique field names from the current batch of data
        data_fields = set()
        for record in data:
            data_fields.update(record.keys())

        new_fields = data_fields - current_schema_fields

        if new_fields:
            new_schema = list(table.schema)
            print(f"Found {len(new_fields)} new fields in the data: {new_fields}. Adding them to BigQuery schema.")

            for field_name in sorted(list(new_fields)):
                # Default new fields to STRING type.
                new_schema.append(SchemaField(field_name, "STRING", description="Dynamically added by Socrata Ingester"))

            # Update the table schema in BigQuery
            table.schema = new_schema
            bq_client.update_table(table, ["schema"])
            print(f"Successfully updated table schema for {table_ref.table_id}.")

    except Exception as e:
        print(f"WARNING: Failed to update BigQuery schema dynamically: {e}")

def load_data_to_bigquery(data):
    """
    Loads the list of JSON records (data) into the specified BigQuery table
    after ensuring the schema can accommodate the fields in the data.
    """
    # table_ref correctly constructs the full reference using the project/dataset/table IDs
    table_ref = bq_client.dataset(BQ_DATASET_ID).table(BQ_TABLE_ID)
    # Handle potential errors
    max_retries = 3
    delay_seconds = 5

    try:
        # 1. Ensure the Dataset exists
        try:
            # Use the BQ_PROJECT_ID here to ensure dataset creation happens in the defined project
            dataset_ref = bq_client.dataset(BQ_DATASET_ID, project=BQ_PROJECT_ID)
            bq_client.get_dataset(dataset_ref)
        except Exception:
            print(f"Dataset {BQ_DATASET_ID} not found in project {BQ_PROJECT_ID}. Creating it.")
            dataset = bigquery.Dataset(dataset_ref)
            bq_client.create_dataset(dataset)

        # 2. Ensure the Table exists with the CORE schema
        table_exists = False
        try:
            bq_client.get_table(table_ref)
            table_exists = True
        except Exception:
            print(f"Table {BQ_TABLE_ID} not found. Creating table with CORE schema.")
            table = bigquery.Table(table_ref, schema=CORE_BQ_SCHEMA)
            bq_client.create_table(table)
            table_exists = True
            print(f"Created table {BQ_TABLE_ID}.")

        if not table_exists:
            raise RuntimeError("Could not confirm or create BigQuery table.")

        # 2. Dynamic Schema Update (Checks incoming keys against existing schema)
        schema_updated = update_bq_schema_if_needed(table_ref, data)

        # 3. Stream data insertion with retries
        for attempt in range(max_retries):
            # The data passed to insert_rows_json MUST be the cleaned data
            errors = bq_client.insert_rows_json(table_ref, data)

            if not errors:
                print(f"Successfully loaded {len(data)} rows into BigQuery.")
                return True

            # If errors occur AND a schema update just happened, wait and retry.
            if schema_updated and attempt < max_retries - 1:
                print(f"Insertion failed, but schema was recently updated. Retrying in {delay_seconds} seconds (Attempt {attempt + 2}/{max_retries}).")
                time.sleep(delay_seconds)
            else:
                # Permanent insertion error or last retry failed
                print(f"Final attempt failed. Errors inserting rows: {json.dumps(errors, indent=2)}")
                return False

        # 4. Dynamic Schema Update (Checks for new fields in the data before inserting)
        update_bq_schema_if_needed(table_ref, data)

        # 5. Stream data insertion
        errors = bq_client.insert_rows_json(table_ref, data)

        if errors:
            print(f"Errors occurred while inserting rows: {json.dumps(errors, indent=2)}")
            return False
        else:
            print(f"Successfully loaded {len(data)} rows into BigQuery.")
            return True

    except Exception as e:
        print(f"Critical BigQuery Loading Error: {e}")
        return False

def ingest_socrata_data(request):
    """
    Main function triggered by HTTP request (manual or Cloud Scheduler).
    The 'request' parameter is required for an HTTP-triggered function.
    """
    print(f"Starting Socrata data ingestion job at {datetime.now().isoformat()}")

    # 1. Determine the current offset by counting rows in the BigQuery table
    current_offset = get_current_offset()

    # Initialize Socrata Client
    try:
        socrata_client = Socrata(
            SOCRATA_HOST,
            SOCRATA_APP_TOKEN,
            timeout=300 # Set a generous timeout for the API call
        )
        if SOCRATA_APP_TOKEN:
            print(f"Socrata Client initialized for host: {SOCRATA_HOST} (Authenticated)")
        else:
            print(f"Socrata Client initialized for host: {SOCRATA_HOST} (Unauthenticated)")

    except Exception as e:
        print(f"Error initializing Socrata client: {e}")
        return "Failed to initialize Socrata client.", 500 # Return HTTP error on failure


    # 2. Construct the Socrata API query using the offset
    query_params = {
        '$limit': CHUNK_SIZE,
        '$offset': current_offset,
        '$order': f'{ORDER_BY_FIELD} ASC', # Ensure a consistent order for continuous loading
    }

    # Apply the manual Socrata filter if configured
    if SOCRATA_WHERE_CLAUSE:
        query_params['$where'] = SOCRATA_WHERE_CLAUSE
        print(f"Applying manual filter: {query_params['$where']}")

    print(f"Fetching data: limit={CHUNK_SIZE}, offset={current_offset}, dataset={SOCRATA_DATASET_ID}")

    try:
        # Fetch the data chunk using the sodapy client
        data = socrata_client.get(
            SOCRATA_DATASET_ID,
            **query_params
        )
        socrata_client.close() # Close the client connection after use

    except Exception as e:
        print(f"API Fetch Error using sodapy: {e}")
        return "Failed to fetch data from Socrata API.", 500 # Return HTTP error on failure

    rows_fetched = len(data)

    # 3. Process the returned data
    if rows_fetched > 0:
        print(f"Successfully fetched {rows_fetched} rows. Cleaning and loading into BigQuery...")

        # Pre-process the data for field name cleanup and removing metadata fields
        cleaned_data = [clean_record(record) for record in data]

        # Load the cleaned data.
        if not load_data_to_bigquery(cleaned_data):
             return "Data loading failed in BigQuery. Check logs for details.", 500
        # Note: The new offset is automatically reflected in the next BQ COUNT(*) query.

    else:
        # 4. Handle the end of the data set
        print("API returned 0 rows. The data set is caught up to the latest available records.")

    print("Ingestion function completed successfully.")

    # Return HTTP success
    return "Ingestion job completed successfully.", 200

## load-cafeteria-inspections main.py

In [ ]:
import os
import json
from datetime import datetime
from google.cloud import bigquery
from google.cloud.bigquery import SchemaField
from sodapy import Socrata
import time

# --- Configuration Constants ---
SOCRATA_HOST = 'data.cityofnewyork.us'
SOCRATA_DATASET_ID = '5ery-qagt' # The resource ID for NYC 311 Service Requests
# WARNING: Temporarily reduced for manual testing to avoid 'request too large' error due to log volume.
# For production via Cloud Scheduler, you may decide to increase or decrease this value.
CHUNK_SIZE = 5000
ORDER_BY_FIELD = 'inspectiondate' # Field to order by for initial load consistency

# Define the custom Socrata Query Language (SoQL) $where clause here.
# Example 1 (DOT only): SOCRATA_WHERE_CLAUSE = "agency = 'DOT'"
# Example 2 (Complex filter): SOCRATA_WHERE_CLAUSE = "agency = 'DOT' AND complaint_type = 'Street Light Out'"
# Set to None or empty string ('') to ingest all data without filtering.
SOCRATA_WHERE_CLAUSE = None # No filter by default, ingest all records. Customize as needed.

# SECRET: Read the Socrata App Token from the Cloud Function environment variables
# Note: This is required for reliable API access.
SOCRATA_APP_TOKEN = os.environ.get('GP5yvTTPsMKDFu2HFlQ4WNcXg')

# BigQuery Configuration
# CRITICAL: You MUST manually replace 'YOUR_PROJECT_ID_HERE' with your actual
# Google Cloud Project ID where BigQuery is hosted.
BQ_PROJECT_ID = 'preston-cis-9440-s26'
BQ_DATASET_ID = 'final_project_raw_ci'
BQ_TABLE_ID = 'source_cafeteria_inspections'

# Initialize BigQuery Client
# When BQ_PROJECT_ID is specified, the client is initialized for that project.
bq_client = bigquery.Client(project=BQ_PROJECT_ID)


# Define the CORE BigQuery schema explicitly.
# This list is used for INITIAL table creation and for type enforcement (TIMESTAMP, FLOAT, STRING for incident_zip).
CORE_BQ_SCHEMA = [
    SchemaField("entityid", "STRING", mode="REQUIRED", description="Unique identifier for the establishment (school)"),
    SchemaField("schoolname", "STRING", description="Establishment (school) name"),
    SchemaField("number", "STRING", description="Building number for establishment (school) location"),
    SchemaField("street", "STRING", description="Street name for establishment (school) location"),
    SchemaField("city", "STRING", description="Municipality of establishment (school) location"),
    SchemaField("state", "STRING", description="State of establishment (school)"),
    SchemaField("borough", "STRING", description="Borough of establishment (school) location"),
    SchemaField("zipcode", "STRING", description="Zip code of establishment (school) location"),
    SchemaField("lastinspection", "TIMESTAMP", description="Date of last inspection"),
    SchemaField("permittee", "STRING", description="Name of business owner holding the permit"),
    SchemaField("inspectiondate", "TIMESTAMP", description="Date an inspection was conducted"),
    SchemaField("ptet", "STRING", description="Permit Type Establishment Type"),
    SchemaField("site_type", "STRING", description="Description of the type of food service establishment (FSE)"),
    SchemaField("level", "STRING", description="Indicator of the type of violation"),
    SchemaField("code", "STRING", description="Violation code associated with inspection finding"),
    SchemaField("violationdescription", "STRING", description="Violation description associated with inspection finding"),
    SchemaField("latitude", "FLOAT", description="Latitude"),
    SchemaField("longitude", "FLOAT", description="Longitude"),
    SchemaField("communityboard", "STRING", description="Community Board"),
    SchemaField("councildistrict", "STRING", description="NYC Council District"),
    SchemaField("censustract", "STRING", description="Census Tract"),
    SchemaField("bin", "STRING", description="Building Identification Number"),
    SchemaField("bbl", "STRING", description="Borough-Block-Lot"),
    SchemaField("nta", "STRING", description="Neighborhood Tabulation Area"),
    SchemaField("borocode", "FLOAT", description="Numerical code for the borough of establishment (school) location"),
]

def clean_record(record):
    """
    Cleans up type anomalies (e.g., empty strings to None), removes Socrata
    internal metadata fields, and cleans field names for BigQuery compatibility.
    """
    cleaned_record = {}

    for key, value in record.items():
        # 1. Drop Socrata internal/metadata and computed region fields
        if key.startswith(':') or key in ['_id', '_submission_details', 'location'] or '__computed_region_' in key:
            continue

        # 2. Clean field names for BigQuery (replace problematic characters AND force lowercase)
        # Force lower() before replacing characters to ensure consistency.
        clean_key = key.lower().replace(':', '_').replace('@', '_').replace(' ', '_')

        # 3. Type normalization: Convert empty strings to None for proper NULL handling in BQ
        if value == "":
            cleaned_record[clean_key] = None
        else:
            cleaned_record[clean_key] = value

        # 4. Special handling for zip codes (FORCES STRING CONVERSION)
        # Ensures leading zeros are preserved and BigQuery receives a string.
        if 'zip' in clean_key.lower() and cleaned_record.get(clean_key) is not None:
             cleaned_record[clean_key] = str(cleaned_record[clean_key])

    return cleaned_record

def get_current_offset():
    """
    Reads the current row count from the BigQuery table. This count serves as the
    starting offset for the Socrata API call.
    """
    # Note: We must explicitly use the BQ_PROJECT_ID here since we are querying it.
    query = f"""
        SELECT COUNT(*) AS current_row_count
        FROM `{BQ_PROJECT_ID}.{BQ_DATASET_ID}.{BQ_TABLE_ID}`
    """
    print(f"Executing BQ query to find current row count (offset).")

    try:
        query_job = bq_client.query(query)
        results = query_job.result()

        for row in results:
            # The count will be 0 if the table is empty
            offset = row.current_row_count
            print(f"Retrieved current offset (row count) from BQ: {offset}")
            return offset

    except Exception as e:
        # If the table doesn't exist yet, the query will fail. We treat this as offset 0.
        print(f"Warning: Failed to retrieve count from BQ (likely table/dataset doesn't exist yet): {e}")

    # Default offset to 0 if BQ query fails (triggers full historical load)
    default_offset = 0
    print(f"Defaulting offset to {default_offset}. This will load historical data.")
    return default_offset

def update_bq_schema_if_needed(table_ref, data):
    """
    Checks the incoming data chunk against the current BigQuery table schema
    and adds any new fields found in the data to the table schema.
    """
    try:
        # Get the current table object and schema
        table = bq_client.get_table(table_ref)
        current_schema_fields = {field.name for field in table.schema}

        # Get all unique field names from the current batch of data
        data_fields = set()
        for record in data:
            data_fields.update(record.keys())

        new_fields = data_fields - current_schema_fields

        if new_fields:
            new_schema = list(table.schema)
            print(f"Found {len(new_fields)} new fields in the data: {new_fields}. Adding them to BigQuery schema.")

            for field_name in sorted(list(new_fields)):
                # Default new fields to STRING type.
                new_schema.append(SchemaField(field_name, "STRING", description="Dynamically added by Socrata Ingester"))

            # Update the table schema in BigQuery
            table.schema = new_schema
            bq_client.update_table(table, ["schema"])
            print(f"Successfully updated table schema for {table_ref.table_id}.")

    except Exception as e:
        print(f"WARNING: Failed to update BigQuery schema dynamically: {e}")

def load_data_to_bigquery(data):
    """
    Loads the list of JSON records (data) into the specified BigQuery table
    after ensuring the schema can accommodate the fields in the data.
    """
    # table_ref correctly constructs the full reference using the project/dataset/table IDs
    table_ref = bq_client.dataset(BQ_DATASET_ID).table(BQ_TABLE_ID)
    # Handle potential errors
    max_retries = 3
    delay_seconds = 5

    try:
        # 1. Ensure the Dataset exists
        try:
            # Use the BQ_PROJECT_ID here to ensure dataset creation happens in the defined project
            dataset_ref = bq_client.dataset(BQ_DATASET_ID, project=BQ_PROJECT_ID)
            bq_client.get_dataset(dataset_ref)
        except Exception:
            print(f"Dataset {BQ_DATASET_ID} not found in project {BQ_PROJECT_ID}. Creating it.")
            dataset = bigquery.Dataset(dataset_ref)
            bq_client.create_dataset(dataset)

        # 2. Ensure the Table exists with the CORE schema
        table_exists = False
        try:
            bq_client.get_table(table_ref)
            table_exists = True
        except Exception:
            print(f"Table {BQ_TABLE_ID} not found. Creating table with CORE schema.")
            table = bigquery.Table(table_ref, schema=CORE_BQ_SCHEMA)
            bq_client.create_table(table)
            table_exists = True
            print(f"Created table {BQ_TABLE_ID}.")

        if not table_exists:
            raise RuntimeError("Could not confirm or create BigQuery table.")

        # 2. Dynamic Schema Update (Checks incoming keys against existing schema)
        schema_updated = update_bq_schema_if_needed(table_ref, data)

        # 3. Stream data insertion with retries
        for attempt in range(max_retries):
            # The data passed to insert_rows_json MUST be the cleaned data
            errors = bq_client.insert_rows_json(table_ref, data)

            if not errors:
                print(f"Successfully loaded {len(data)} rows into BigQuery.")
                return True

            # If errors occur AND a schema update just happened, wait and retry.
            if schema_updated and attempt < max_retries - 1:
                print(f"Insertion failed, but schema was recently updated. Retrying in {delay_seconds} seconds (Attempt {attempt + 2}/{max_retries}).")
                time.sleep(delay_seconds)
            else:
                # Permanent insertion error or last retry failed
                print(f"Final attempt failed. Errors inserting rows: {json.dumps(errors, indent=2)}")
                return False

        # 4. Dynamic Schema Update (Checks for new fields in the data before inserting)
        update_bq_schema_if_needed(table_ref, data)

        # 5. Stream data insertion
        errors = bq_client.insert_rows_json(table_ref, data)

        if errors:
            print(f"Errors occurred while inserting rows: {json.dumps(errors, indent=2)}")
            return False
        else:
            print(f"Successfully loaded {len(data)} rows into BigQuery.")
            return True

    except Exception as e:
        print(f"Critical BigQuery Loading Error: {e}")
        return False

def ingest_socrata_data(request):
    """
    Main function triggered by HTTP request (manual or Cloud Scheduler).
    The 'request' parameter is required for an HTTP-triggered function.
    """
    print(f"Starting Socrata data ingestion job at {datetime.now().isoformat()}")

    # 1. Determine the current offset by counting rows in the BigQuery table
    current_offset = get_current_offset()

    # Initialize Socrata Client
    try:
        socrata_client = Socrata(
            SOCRATA_HOST,
            SOCRATA_APP_TOKEN,
            timeout=300 # Set a generous timeout for the API call
        )
        if SOCRATA_APP_TOKEN:
            print(f"Socrata Client initialized for host: {SOCRATA_HOST} (Authenticated)")
        else:
            print(f"Socrata Client initialized for host: {SOCRATA_HOST} (Unauthenticated)")

    except Exception as e:
        print(f"Error initializing Socrata client: {e}")
        return "Failed to initialize Socrata client.", 500 # Return HTTP error on failure


    # 2. Construct the Socrata API query using the offset
    query_params = {
        '$limit': CHUNK_SIZE,
        '$offset': current_offset,
        '$order': f'{ORDER_BY_FIELD} ASC', # Ensure a consistent order for continuous loading
    }

    # Apply the manual Socrata filter if configured
    if SOCRATA_WHERE_CLAUSE:
        query_params['$where'] = SOCRATA_WHERE_CLAUSE
        print(f"Applying manual filter: {query_params['$where']}")

    print(f"Fetching data: limit={CHUNK_SIZE}, offset={current_offset}, dataset={SOCRATA_DATASET_ID}")

    try:
        # Fetch the data chunk using the sodapy client
        data = socrata_client.get(
            SOCRATA_DATASET_ID,
            **query_params
        )
        socrata_client.close() # Close the client connection after use

    except Exception as e:
        print(f"API Fetch Error using sodapy: {e}")
        return "Failed to fetch data from Socrata API.", 500 # Return HTTP error on failure

    rows_fetched = len(data)

    # 3. Process the returned data
    if rows_fetched > 0:
        print(f"Successfully fetched {rows_fetched} rows. Cleaning and loading into BigQuery...")

        # Pre-process the data for field name cleanup and removing metadata fields
        cleaned_data = [clean_record(record) for record in data]

        # Load the cleaned data.
        if not load_data_to_bigquery(cleaned_data):
             return "Data loading failed in BigQuery. Check logs for details.", 500
        # Note: The new offset is automatically reflected in the next BQ COUNT(*) query.

    else:
        # 4. Handle the end of the data set
        print("API returned 0 rows. The data set is caught up to the latest available records.")

    print("Ingestion function completed successfully.")

    # Return HTTP success
    return "Ingestion job completed successfully.", 200

# dbt.com
## YAML

In [ ]:
version: 2
sources:
  - name: raw_311
    description: Raw data loaded from NYC Open Data
    database: preston-cis-9440-s26
    schema: final_project_raw_sr
    tables:
      - name: source_service_request
        description: |
          311 Service Requests from 2020 to Present.
          One row per service request.
        columns:
          - name: address_type
            description: Type of incident location information available.
            tests:
              - not_null
          - name: agency
            description: Acronym of responding City Government Agency
            tests:
              - not_null
          - name: agency_name
            description: Full Agency name of responding City Government Agency
            tests:
              - not_null
          - name: bbl
            description: Borough Block and Lot, provided by geovalidation. Parcel number to identify the location of location of buildings and properties in NYC.
          - name: borough
            description: Provided by the submitter and confirmed by geovalidation.
          - name: city
            description: City of the incident location provided by geovalidation.
          - name: closed_date
            description: Date SR was closed by responding agency
          - name: community_board
            description: Provided by geovalidation.
          - name: complaint_type
            description: This is the first level of a hierarchy identifying the topic of the incident or condition. Problem (formerly Complaint Type) broadly describes the topic of the incident or condition and are defined by the responding agencies. New Problems may be added in response to changes in customer demand.
            tests:
              - not_null
          - name: council_district
            description: The City Council district where the service request is located.
          - name: created_date
            description: Date SR was created
            tests:
              - not_null
          - name: cross_street_1
            description: First Cross street based on the geo validated incident location
          - name: cross_street_2
            description: Second Cross Street based on the geo validated incident location
          - name: descriptor
            description: This is associated to the Problem (formerly Complaint Type), and provides further detail on the incident or condition. Problem Detail (formerly Descriptor) values are dependent on the Problem, and are not always required in the service request.
          - name: descriptor_2
            description: A third level of detail about the Problem (formerly Complaint Type) beyond the Problem Detail (formerly Descriptor). This is not used by every category of issue.
          #due_date would go here, but all data pulled didn't have anything in this column
          - name: incident_address
            description: House number of incident address provided by submitter.
          - name: incident_zip
            description: Incident location zip code, provided by geo validation.
          - name: intersection_street_1
            description: First intersecting street based on geo validated incident location
          - name: intersection_street_2
            description: Second intersecting street based on geo validated incident location
          - name: landmark
            description: If the incident location is identified as a Landmark the name of the landmark will display here
          - name: latitude
            description: Geo based Lat of the incident location
          #location would go here, but as it is geospatial "Point" data it wasn't captured by a table
          - name: location_type
            description: Describes the type of location used in the address information
          - name: longitude
            description: Geo based Long of the incident location
          - name: open_data_channel_type
            description: Indicates how the SR was submitted to 311. i.e. By Phone, Online, Mobile, Other or Unknown.
            tests:
              - not_null
          - name: park_borough
            description: The borough of incident if it is a Parks Dept facility
          - name: park_facility_name
            description: If the incident location is a Parks Dept facility, the Name of the facility will appear here
          - name: police_precinct
            description: The NYPD precinct where the service request is located.
          - name: resolution_action_updated_date
            description: Date when responding agency last updated the SR.
          - name: resolution_description
            description: Describes the last action taken on the SR by the responding agency. May describe next or future steps.
          #road_ramp returned no data
          - name: status
            description: Status of SR submitted
            tests:
              - not_null
          - name: street_name
            description: Street name of incident address provided by the submitter
          #variables with no data:
          #taxi_company_borough
          #taxi_pick_up_location
          #vehicle_type
          - name: unique_key
            description: Unique ID for the service request
            tests:
              - unique
              - not_null
          - name: x_coordinate_state_plane
            description: Geo validated, X coordinate of the incident location.
          - name: y_coordinate_state_plane
            description: Geo validated, Y coordinate of the incident location.
          - name: facility_type
            description: If available, this field describes the type of city facility associated to the SR


  - name: raw_inspections
    description: Raw data loaded from NYC Open Data
    database: preston-cis-9440-s26
    schema: final_project_raw_ci
    tables:
      - name: source_cafeteria_inspections
        description: |
          DOHMH School Cafeteria Inspections (2020-Present)
          One row per application. Note: original dataset may have two or more entries for the same restaurant.
        columns:
          - name: bbl
            description: Borough-Block-Lot
            tests:
              - not_null
          - name: bin
            description: Building Identification Number
            tests:
              - not_null
          - name: borocode
            description: Numerical code for the borough of establishment (school) location
            tests:
              - not_null
          - name: borough
            description: Borough of establishment (school) location
            tests:
              - not_null
          - name: censustract
            description: Census Tract
            tests:
              - not_null
          - name: city
            description: Municipality of establishment (school)
            tests:
              - not_null
          - name: code
            description: Violation code associated with inspection finding; Blank field indicates no violations were cited during inspection
          - name: communityboard
            description: Community Board
            tests:
              - not_null
          - name: councildistrict
            description: NYC Council District
            tests:
              - not_null
          - name: entityid
            description: Unique identifier for the establishment (school)
            tests:
              - unique
              - not_null
          - name: inspectiondate
            description: Date an inspection was conducted
            tests:
              - not_null
          - name: lastinspection
            description: Date of last inspection
            tests:
              - not_null
          - name: latitude
            description: Latitude
            tests:
              - not_null
          - name: level
            description: Indicator of the type of violation; Critical violations are those most likely to contribute to foodborne illness.
          - name: longitude
            description: Longitude
            tests:
              - not_null
          - name: nta
            description: Neighborhood Tabulation Area
            tests:
              - not_null
          - name: number
            description: Building number for establishment (school) location
            tests:
              - not_null
          - name: permittee
            description: Name of business owner holding the permit
            tests:
              - not_null
          - name: ptet
            description: Permit Type Establishment Type - The four digit code indicating the type of permit and the type of food service establishment (FSE)
            tests:
              - not_null
          - name: schoolname
            description: Establishment (school) name
            tests:
              - not_null
          - name: site_type
            description: The description of the type of food service establishment (FSE)
            tests:
              - not_null
          - name: state
            description: State of establishment (school)
            tests:
              - not_null
          - name: street
            description: Street name for establishment (school) location
            tests:
              - not_null
          - name: violationdescription
            description: Violation description associated with inspection finding; Blank field indicates no violations were cited during inspection
          - name: zipcode
            description: Zip code of establishment (school) location
            tests:
              - not_null

## stg_cafeteria_inspections

In [ ]:
-- Clean and standardize cafeteria inspections raw data
-- One row per establishment (school) per inspection (composite key)

WITH source AS (
    SELECT *
    FROM {{ source('raw_inspections', 'source_cafeteria_inspections') }}
),

typed AS (
    SELECT
        -- Select transformed fields explicitly to avoid star/except expansion issues
        -- IDs and school info
        NULLIF(TRIM(CAST(entityid AS STRING)), '') AS record_id,
        NULLIF(REGEXP_REPLACE(TRIM(CAST(schoolname AS STRING)), r'\s+', ' '), '') AS school_name,
        NULLIF(REGEXP_REPLACE(TRIM(CAST(number AS STRING)), r'\s+', ' '), '') AS building_number,
        NULLIF(REGEXP_REPLACE(TRIM(CAST(street AS STRING)), r'\s+', ' '), '') AS street_name,
        NULLIF(REGEXP_REPLACE(TRIM(CAST(city AS STRING)), r'\s+', ' '), '') AS city,
        UPPER(NULLIF(REGEXP_REPLACE(TRIM(CAST(state AS STRING)), r'\s+', ' '), '')) AS state,

        -- Borough normalization for cleaner dimensional joins
        CASE
            WHEN UPPER(REGEXP_REPLACE(TRIM(CAST(borough AS STRING)), r'\s+', ' ')) IN ('MANHATTAN', 'NEW YORK COUNTY') THEN 'Manhattan'
            WHEN UPPER(REGEXP_REPLACE(TRIM(CAST(borough AS STRING)), r'\s+', ' ')) IN ('BRONX', 'THE BRONX') THEN 'Bronx'
            WHEN UPPER(REGEXP_REPLACE(TRIM(CAST(borough AS STRING)), r'\s+', ' ')) IN ('BROOKLYN', 'KINGS COUNTY') THEN 'Brooklyn'
            WHEN UPPER(REGEXP_REPLACE(TRIM(CAST(borough AS STRING)), r'\s+', ' ')) IN ('QUEENS', 'QUEEN', 'QUEENS COUNTY') THEN 'Queens'
            WHEN UPPER(REGEXP_REPLACE(TRIM(CAST(borough AS STRING)), r'\s+', ' ')) IN ('STATEN ISLAND', 'RICHMOND COUNTY') THEN 'Staten Island'
            WHEN NULLIF(REGEXP_REPLACE(TRIM(CAST(borough AS STRING)), r'\s+', ' '), '') IS NULL THEN NULL
            ELSE INITCAP(REGEXP_REPLACE(TRIM(CAST(borough AS STRING)), r'\s+', ' '))
        END AS borough,

        -- Zip code cleanup
        CASE
            WHEN NULLIF(TRIM(CAST(zipcode AS STRING)), '') IS NULL THEN NULL
            WHEN REGEXP_CONTAINS(TRIM(CAST(zipcode AS STRING)), r'^\d{5}$') THEN TRIM(CAST(zipcode AS STRING))
            WHEN REGEXP_CONTAINS(TRIM(CAST(zipcode AS STRING)), r'^\d{5}-\d{4}$') THEN TRIM(CAST(zipcode AS STRING))
            ELSE NULL
        END AS zip_code,

        NULLIF(REGEXP_REPLACE(TRIM(CAST(permittee AS STRING)), r'\s+', ' '), '') AS permittee,
        NULLIF(REGEXP_REPLACE(TRIM(CAST(ptet AS STRING)), r'\s+', ' '), '') AS ptet,
        NULLIF(REGEXP_REPLACE(TRIM(CAST(site_type AS STRING)), r'\s+', ' '), '') AS site_type,
        NULLIF(REGEXP_REPLACE(TRIM(CAST(level AS STRING)), r'\s+', ' '), '') AS violation_level,
        NULLIF(REGEXP_REPLACE(TRIM(CAST(code AS STRING)), r'\s+', ' '), '') AS violation_code,
        NULLIF(REGEXP_REPLACE(TRIM(CAST(violationdescription AS STRING)), r'\s+', ' '), '') AS violation_description,

        -- Geography and governance fields
        NULLIF(REGEXP_REPLACE(TRIM(CAST(communityboard AS STRING)), r'\s+', ' '), '') AS community_board,
        NULLIF(REGEXP_REPLACE(TRIM(CAST(councildistrict AS STRING)), r'\s+', ' '), '') AS council_district,
        NULLIF(REGEXP_REPLACE(TRIM(CAST(censustract AS STRING)), r'\s+', ' '), '') AS census_tract,
        NULLIF(REGEXP_REPLACE(TRIM(CAST(bin AS STRING)), r'\s+', ' '), '') AS bin,
        NULLIF(REGEXP_REPLACE(TRIM(CAST(bbl AS STRING)), r'\s+', ' '), '') AS bbl,
        NULLIF(REGEXP_REPLACE(TRIM(CAST(nta AS STRING)), r'\s+', ' '), '') AS nta,

        -- Parse timestamps from raw source
        SAFE_CAST(inspectiondate AS TIMESTAMP) AS inspection_ts,
        SAFE_CAST(lastinspection AS TIMESTAMP) AS last_inspection_ts,

        SAFE_CAST(latitude AS DECIMAL) AS latitude,
        SAFE_CAST(longitude AS DECIMAL) AS longitude,
        SAFE_CAST(borocode AS STRING) AS borocode
    FROM source
),

cleaned AS (
    SELECT
        record_id,
        school_name,
        building_number,
        street_name,
        city,
        state,
        borough,
        zip_code,

        -- Split datetime into separate fields for reporting flexibility (keep date only)
        DATE(inspection_ts) AS inspection_date,
        DATE(last_inspection_ts) AS last_inspection_date,

        permittee,
        ptet,
        site_type,
        violation_level,
        violation_code,
        violation_description,

        latitude,
        longitude,
        community_board,
        council_district,
        census_tract,
        bin,
        bbl,
        nta,
        borocode,

        CURRENT_TIMESTAMP() AS ci_stg_loaded_at
    FROM typed
    WHERE record_id IS NOT NULL
),

deduped AS (
    SELECT *
    FROM cleaned
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY
            record_id, school_name, building_number, street_name, city, state, borough, zip_code,
            inspection_date, last_inspection_date,
            permittee, ptet, site_type, violation_level, violation_code, violation_description,
            latitude, longitude, community_board, council_district, census_tract, bin, bbl, nta, borocode
        ORDER BY ci_stg_loaded_at DESC
    ) = 1
)

SELECT *
FROM deduped

## stg_nyc_311_doedohmh

In [ ]:
-- Clean and standardize NYC 311 service request data
-- One row per service request, identified by unique_key
-- Deduplication: keep most recent by created_date if duplicates exist (unlikely)

WITH source AS (
   SELECT *
   FROM {{ source('raw_311', 'source_service_request') }}
),

typed AS (
   SELECT
       -- Select transformed fields explicitly to avoid star/except expansion issues
       -- IDs and core identifiers
       CAST(unique_key AS STRING) AS service_request_id,
       CAST(facility_type AS STRING) AS facility_type,

       -- Agency information
       CAST(agency AS STRING) AS agency,
       CAST(agency_name AS STRING) AS agency_name,
       CAST(complaint_type AS STRING) AS complaint_type,
       CAST(descriptor AS STRING) AS descriptor,
       CAST(descriptor_2 AS STRING) AS descriptor_2,
       CAST(status AS STRING) AS status,
       CAST(open_data_channel_type AS STRING) AS open_data_channel_type,

       -- Resolution tracking
       CAST(resolution_description AS STRING) AS resolution_description,

       -- Location type and address fields
       CAST(address_type AS STRING) AS address_type,
       CAST(location_type AS STRING) AS location_type,
       NULLIF(REGEXP_REPLACE(TRIM(CAST(incident_address AS STRING)), r'\s+', ' '), '') AS incident_address,
       NULLIF(REGEXP_REPLACE(TRIM(CAST(street_name AS STRING)), r'\s+', ' '), '') AS street_name,
       NULLIF(REGEXP_REPLACE(TRIM(CAST(cross_street_1 AS STRING)), r'\s+', ' '), '') AS cross_street_1,
       NULLIF(REGEXP_REPLACE(TRIM(CAST(cross_street_2 AS STRING)), r'\s+', ' '), '') AS cross_street_2,
       NULLIF(REGEXP_REPLACE(TRIM(CAST(intersection_street_1 AS STRING)), r'\s+', ' '), '') AS intersection_street_1,
       NULLIF(REGEXP_REPLACE(TRIM(CAST(intersection_street_2 AS STRING)), r'\s+', ' '), '') AS intersection_street_2,
       NULLIF(REGEXP_REPLACE(TRIM(CAST(landmark AS STRING)), r'\s+', ' '), '') AS landmark,

       -- Borough normalization for cleaner dimensional joins
       CASE
           WHEN UPPER(REGEXP_REPLACE(TRIM(CAST(borough AS STRING)), r'\s+', ' ')) IN ('MANHATTAN', 'NEW YORK COUNTY') THEN 'Manhattan'
           WHEN UPPER(REGEXP_REPLACE(TRIM(CAST(borough AS STRING)), r'\s+', ' ')) IN ('BRONX', 'THE BRONX') THEN 'Bronx'
           WHEN UPPER(REGEXP_REPLACE(TRIM(CAST(borough AS STRING)), r'\s+', ' ')) IN ('BROOKLYN', 'KINGS COUNTY') THEN 'Brooklyn'
           WHEN UPPER(REGEXP_REPLACE(TRIM(CAST(borough AS STRING)), r'\s+', ' ')) IN ('QUEENS', 'QUEEN', 'QUEENS COUNTY') THEN 'Queens'
           WHEN UPPER(REGEXP_REPLACE(TRIM(CAST(borough AS STRING)), r'\s+', ' ')) IN ('STATEN ISLAND', 'RICHMOND COUNTY') THEN 'Staten Island'
           WHEN NULLIF(REGEXP_REPLACE(TRIM(CAST(borough AS STRING)), r'\s+', ' '), '') IS NULL THEN NULL
           ELSE INITCAP(REGEXP_REPLACE(TRIM(CAST(borough AS STRING)), r'\s+', ' '))
       END AS borough,

       CAST(city AS STRING) AS city,

       -- Zip code cleanup
       CASE
           WHEN NULLIF(TRIM(CAST(incident_zip AS STRING)), '') IS NULL THEN NULL
           WHEN REGEXP_CONTAINS(TRIM(CAST(incident_zip AS STRING)), r'^\d{5}$') THEN TRIM(CAST(incident_zip AS STRING))
           WHEN REGEXP_CONTAINS(TRIM(CAST(incident_zip AS STRING)), r'^\d{5}-\d{4}$') THEN TRIM(CAST(incident_zip AS STRING))
           ELSE NULL
       END AS incident_zip,

       -- Parks-related fields
       CAST(park_borough AS STRING) AS park_borough,
       CAST(park_facility_name AS STRING) AS park_facility_name,

       -- Geography and governance fields
       CAST(community_board AS STRING) AS community_board,
       CAST(council_district AS STRING) AS council_district,
       CAST(police_precinct AS STRING) AS police_precinct,
       CAST(bbl AS STRING) AS bbl,

       -- Timestamps: split into DATE and TIME
       SAFE_CAST(created_date AS TIMESTAMP) AS created_ts,
       SAFE_CAST(closed_date AS TIMESTAMP) AS closed_ts,
       SAFE_CAST(resolution_action_updated_date AS TIMESTAMP) AS resolution_action_updated_ts,

       -- Coordinates
       SAFE_CAST(latitude AS DECIMAL) AS latitude,
       SAFE_CAST(longitude AS DECIMAL) AS longitude,
       SAFE_CAST(x_coordinate_state_plane AS DECIMAL) AS x_coordinate_state_plane,
       SAFE_CAST(y_coordinate_state_plane AS DECIMAL) AS y_coordinate_state_plane

   FROM source
),

cleaned AS (
   SELECT
       service_request_id,
       facility_type,
       agency,
       agency_name,
       complaint_type,
       descriptor,
       descriptor_2,
       status,
       open_data_channel_type,
       resolution_description,

       address_type,
       location_type,
       incident_address,
       street_name,
       cross_street_1,
       cross_street_2,
       intersection_street_1,
       intersection_street_2,
       landmark,

       borough,
       city,
       incident_zip,

       park_borough,
       park_facility_name,

       community_board,
       council_district,
       police_precinct,
       bbl,

       -- Split timestamps into DATE and TIME for reporting flexibility
       DATE(created_ts) AS created_date,
       TIME(created_ts) AS created_time,
       DATE(closed_ts) AS closed_date,
       TIME(closed_ts) AS closed_time,
       DATE(resolution_action_updated_ts) AS resolution_action_updated_date,
       TIME(resolution_action_updated_ts) AS resolution_action_updated_time,

       latitude,
       longitude,
       x_coordinate_state_plane,
       y_coordinate_state_plane,

       CURRENT_TIMESTAMP() AS sr_stg_loaded_at

   FROM typed
   WHERE service_request_id IS NOT NULL
),

deduped AS (
   SELECT *
   FROM cleaned
   -- Keep most recent by created_date if exact duplicates exist (unique_key should be unique)
   QUALIFY ROW_NUMBER() OVER (
       PARTITION BY service_request_id
       ORDER BY created_date DESC, created_time DESC
   ) = 1
)

SELECT *
FROM deduped
-- Suggested model name: stg_nyc_311_service_requests

# Dimension Tables

Table dim_problem {
  problem_sk               int      [pk]
  complaint_type           varchar  [null]
  descriptor               varchar  [null]
  open_data_channel_type   varchar  [null]
  status                   varchar  [null]
}

In [ ]:
WITH source AS (
   SELECT DISTINCT
       complaint_type,
       descriptor,
       open_data_channel_type,
       status
   FROM {{ ref('stg_nyc_311_doedohmh') }}
)

SELECT
   {{ dbt_utils.generate_surrogate_key(['complaint_type', 'descriptor', 'open_data_channel_type', 'status']) }} AS problem_sk,
   complaint_type,
   descriptor,
   open_data_channel_type,
   status
FROM source

Table dim_sr_government {
  sr_gov_sk            int      [pk]
  agency               varchar  [null]
  agency_name          varchar  [null]
  sr_community_board   varchar  [null]
  police_precinct      varchar  [null]
}


In [ ]:
WITH source AS (
   SELECT DISTINCT
       agency,
       agency_name,
       community_board,
       police_precinct
   FROM {{ ref('stg_nyc_311_doedohmh') }}
)

SELECT
   {{ dbt_utils.generate_surrogate_key(['agency', 'agency_name', 'community_board', 'police_precinct']) }} AS sr_gov_sk,
   agency,
   agency_name,
   community_board AS sr_community_board,
   police_precinct
FROM source

Table dim_sr_geography {
  sr_geo_sk                int      [pk]
  incident_address         varchar  [null]
  street_name              varchar  [null]
  cross_street_1           varchar  [null]
  cross_street_2           varchar  [null]
  intersection_street_1    varchar  [null]
  intersection_street_2    varchar  [null]
  address_type             varchar  [null]
  sr_city                  varchar  [null]
  landmark                 varchar  [null]
  sr_bbl                   bigint   [null]
}

In [ ]:
WITH source AS (
   SELECT DISTINCT
       incident_address,
       street_name,
       cross_street_1,
       cross_street_2,
       intersection_street_1,
       intersection_street_2,
       address_type,
       city AS sr_city,
       landmark,
       bbl AS sr_bbl
   FROM {{ ref('stg_nyc_311_doedohmh') }}
)

SELECT
   {{ dbt_utils.generate_surrogate_key(['incident_address', 'street_name', 'cross_street_1', 'cross_street_2', 'intersection_street_1', 'intersection_street_2', 'address_type', 'sr_city', 'landmark', 'sr_bbl']) }} AS sr_geo_sk,
   incident_address,
   street_name,
   cross_street_1,
   cross_street_2,
   intersection_street_1,
   intersection_street_2,
   address_type,
   sr_city,
   landmark,
   sr_bbl
FROM source

Table dim_school {
  school_sk               int           [pk]
  dbn                     varchar       [null]
  school_name             varchar       [null]
  permittee               varchar       [null]
  ptet                    int           [null]
  site_type               varchar       [null]
  facility_type           varchar       [null]

  school_level            varchar       [null]
  total_enrollment        int           [null]
  economic_need_index     decimal(4,1)  [null]
  academic_year           varchar       [null]

  is_current_school_year  boolean       [null]
  dw_inserted_at          timestamp     [null]
}

In [ ]:
WITH source AS (
   SELECT DISTINCT
       school_name,
       permittee,
       ptet,
       site_type
   FROM {{ ref('stg_cafeteria_inspections') }}
)

SELECT
   {{ dbt_utils.generate_surrogate_key(['school_name']) }} AS school_sk,
   /* Build school dimension using mapping (school name → DBN),
   used for scalability if we need to join with other tables */
   school_name AS dbn,
   school_name,
   permittee,
   ptet,
   site_type,
   CASE
      WHEN UPPER(school_name) LIKE '%ELEMENTARY%' THEN 'ELEMENTARY'
      WHEN UPPER(school_name) LIKE '%MIDDLE%' THEN 'MIDDLE'
      WHEN UPPER(school_name) LIKE '%HIGH%' THEN 'HIGH'
      WHEN UPPER(school_name) LIKE '%PREP%' THEN 'PREP'
      WHEN UPPER(school_name) LIKE '%ACADEMY%' THEN 'ACADEMY'
      ELSE NULL
   END AS school_level,

   /* COMMENTING OUT FOR NOW (per discussion with team) - mapping from
   facility_type to economic_need_index is not available in source,
   present for potential scalability with potential third dataset,
   set to NULL for now, except school_level with temporary workaround.
   NULL AS facility_type,
   NULL AS total_enrollment,
   NULL AS economic_need_index,
   NULL AS academic_year,
   NULL AS is_current_school_year, */
   CURRENT_TIMESTAMP() AS dw_inserted_at
FROM source

Table dim_violation {
  violation_sk             int      [pk]
  level                    varchar  [null]
  code                     varchar  [null]
  violation_description    text     [null]
}

In [ ]:
WITH source AS (
   SELECT DISTINCT
       violation_level,
       violation_code,
       violation_description
   FROM {{ ref('stg_cafeteria_inspections') }}
)

SELECT
   {{ dbt_utils.generate_surrogate_key(['violation_level', 'violation_code']) }} AS violation_sk,
   violation_level AS level,
   violation_code AS code,
   violation_description
FROM source

Table dim_ci_geography {
  ci_geo_sk       int      [pk]
  number          varchar  [null]
  street          varchar  [null]
  ci_city         varchar  [null]
  state           varchar  [null]
  bin             int      [null]
  ci_bbl          bigint   [null]
  nta             varchar  [null]
  borocode        int      [null]
}

In [ ]:
WITH source AS (
   SELECT DISTINCT
       building_number,
       street_name,
       city,
       state,
       bin,
       bbl,
       nta,
       borocode
   FROM {{ ref('stg_cafeteria_inspections') }}
)

SELECT
   {{ dbt_utils.generate_surrogate_key(['building_number', 'street_name', 'city', 'state', 'bin', 'bbl', 'nta', 'borocode']) }} AS ci_geo_sk,
   building_number AS number,
   street_name AS street,
   city AS ci_city,
   state,
   bin,
   bbl AS ci_bbl,
   nta,
   borocode
FROM source

Table dim_ci_government {
  ci_gov_sk           int      [pk]
  community_board     int      [null]
  census_tract        int      [null]
}

In [ ]:
WITH source AS (
   SELECT DISTINCT
       community_board,
       census_tract
   FROM {{ ref('stg_cafeteria_inspections') }}
)

SELECT
   {{ dbt_utils.generate_surrogate_key(['community_board', 'census_tract']) }} AS ci_gov_sk,
   community_board,
   census_tract
FROM source

# Shared Tables

In [ ]:
Table dim_shared_date {
  date_sk         int      [pk]
  full_date       date     [not null]
  year            int      [not null]
  quarter         int      [not null]
  month_num       int      [not null]
  month_name      varchar  [not null]
  day_num         int      [not null]
  day_name        varchar  [not null]
  is_school_day   boolean  [not null]
}


SyntaxError: invalid syntax (1013205554.py, line 1)

In [ ]:
WITH all_dates AS (
   -- Collect all distinct dates used across the mart sources
   SELECT DISTINCT CAST(created_date AS DATE) AS full_date
   FROM {{ ref('stg_nyc_311_doedohmh') }}
   WHERE created_date IS NOT NULL

   UNION DISTINCT

   SELECT DISTINCT CAST(resolution_action_updated_date AS DATE) AS full_date
   FROM {{ ref('stg_nyc_311_doedohmh') }}
   WHERE resolution_action_updated_date IS NOT NULL

   UNION DISTINCT

   SELECT DISTINCT CAST(closed_date AS DATE) AS full_date
   FROM {{ ref('stg_nyc_311_doedohmh') }}
   WHERE closed_date IS NOT NULL

   UNION DISTINCT

   SELECT DISTINCT CAST(inspection_date AS DATE) AS full_date
   FROM {{ ref('stg_cafeteria_inspections') }}
   WHERE inspection_date IS NOT NULL

   UNION DISTINCT

   SELECT DISTINCT CAST(last_inspection_date AS DATE) AS full_date
   FROM {{ ref('stg_cafeteria_inspections') }}
   WHERE last_inspection_date IS NOT NULL
),

date_dimension AS (
   SELECT
       {{ dbt_utils.generate_surrogate_key(['full_date']) }} AS date_sk,
       full_date,
       EXTRACT(YEAR FROM full_date) AS year,
       EXTRACT(QUARTER FROM full_date) AS quarter,
       EXTRACT(MONTH FROM full_date) AS month_num,
       FORMAT_DATE('%B', full_date) AS month_name,
       EXTRACT(DAY FROM full_date) AS day_num,
       FORMAT_DATE('%A', full_date) AS day_name,
       EXTRACT(DAYOFWEEK FROM full_date) NOT IN (1, 7) AS is_school_day
   FROM all_dates
)

SELECT *
FROM date_dimension

Table dim_shared_geography {
  geo_sk             int      [pk]
  borough            varchar  [null]
  zip_code           varchar  [null]
  council_district   int      [null]
}

In [ ]:
WITH geography_rows AS (
   -- Collect distinct shared geography attributes from both marts
   SELECT DISTINCT
       borough,
       incident_zip AS zip_code,
       council_district
   FROM {{ ref('stg_nyc_311_doedohmh') }}
   WHERE borough IS NOT NULL
      OR incident_zip IS NOT NULL
      OR council_district IS NOT NULL

   UNION DISTINCT

   SELECT DISTINCT
       borough,
       zip_code,
       council_district
   FROM {{ ref('stg_cafeteria_inspections') }}
   WHERE borough IS NOT NULL
      OR zip_code IS NOT NULL
      OR council_district IS NOT NULL
),

dim_shared_geography AS (
   SELECT
       {{ dbt_utils.generate_surrogate_key(['borough', 'zip_code', 'council_district']) }} AS geo_sk,
       borough,
       zip_code,
       council_district
   FROM geography_rows
)

SELECT *
FROM dim_shared_geography

# Fact Tables

Table fact_service_request {
  fact_service_request_sk               int        [pk]
  unique_key                            varchar    [not null]

  created_date_sk                       int        [not null]
  due_date_sk                           int        [null]
  updated_date_sk                       int        [null]
  closed_date_sk                        int        [null]

  problem_sk                            int        [not null]
  sr_gov_sk                             int        [not null]
  sr_geo_sk                             int        [null]
  geo_sk                                int        [null]

  count_days_created_to_closed          int        [null]
  count_business_days_created_to_closed int        [null]
  resolution_description                text       [null]

  latitude                              float      [null]
  longitude                             float      [null]
  x_coordinate_state_plane              int        [null]
  y_coordinate_state_plane              int        [null]

  sr_dw_inserted_at                     timestamp  [not null]
  sr_dw_updated_at                      timestamp  [null]
}

In [ ]:
WITH source AS (
   SELECT *
   FROM {{ ref('stg_nyc_311_doedohmh') }}
),

fact_service_request AS (
   SELECT
       {{ dbt_utils.generate_surrogate_key(['service_request_id']) }} AS fact_service_request_sk,
       s.service_request_id AS unique_key,

       d1.date_sk AS created_date_sk,
       NULL AS due_date_sk,
       d2.date_sk AS updated_date_sk,
       d3.date_sk AS closed_date_sk,

       p.problem_sk,
       srg.sr_gov_sk,
       srg2.sr_geo_sk,
       sg.geo_sk,

       CASE
           WHEN s.created_date IS NOT NULL AND s.closed_date IS NOT NULL THEN DATE_DIFF(s.closed_date, s.created_date, DAY)
       END AS count_days_created_to_closed,
       NULL AS count_business_days_created_to_closed,
       s.resolution_description,
       s.latitude,
       s.longitude,
       s.x_coordinate_state_plane,
       s.y_coordinate_state_plane,
       CURRENT_TIMESTAMP() AS sr_dw_inserted_at,
       NULL AS sr_dw_updated_at
   FROM source s
   LEFT JOIN {{ ref('dim_shared_date') }} d1 ON d1.full_date = s.created_date
   LEFT JOIN {{ ref('dim_shared_date') }} d2 ON d2.full_date = s.resolution_action_updated_date
   LEFT JOIN {{ ref('dim_shared_date') }} d3 ON d3.full_date = s.closed_date
   LEFT JOIN {{ ref('dim_problem') }} p ON p.complaint_type = s.complaint_type AND p.descriptor = s.descriptor AND p.open_data_channel_type = s.open_data_channel_type AND p.status = s.status
   LEFT JOIN {{ ref('dim_sr_government') }} srg ON srg.agency = s.agency AND srg.agency_name = s.agency_name AND srg.sr_community_board = s.community_board AND srg.police_precinct = s.police_precinct
   LEFT JOIN {{ ref('dim_sr_geography') }} srg2 ON srg2.incident_address = s.incident_address AND srg2.street_name = s.street_name AND srg2.cross_street_1 = s.cross_street_1 AND srg2.cross_street_2 = s.cross_street_2 AND srg2.intersection_street_1 = s.intersection_street_1 AND srg2.intersection_street_2 = s.intersection_street_2 AND srg2.address_type = s.address_type AND srg2.sr_city = s.city AND srg2.landmark = s.landmark AND srg2.sr_bbl = s.bbl
   LEFT JOIN {{ ref('dim_shared_geography') }} sg ON sg.borough = s.borough AND sg.zip_code = s.incident_zip AND sg.council_district = s.council_district
)

SELECT *
FROM fact_service_request

Table fact_cafeteria_inspection {
  inspection_sk              int        [pk]
  record_id                  int        [not null]

  inspection_date_sk         int        [not null]
  last_inspection_date_sk    int        [null]

  school_sk                  int        [not null]
  violation_sk               int        [null]
  ci_geo_sk                  int        [null]
  ci_gov_sk                  int        [null]
  geo_sk                     int        [null]

  latitude                   float      [null]
  longitude                  float      [null]

  ci_dw_inserted_at          timestamp  [not null]
  ci_dw_updated_at           timestamp  [null]
}

In [ ]:
WITH source AS (
   SELECT *
   FROM {{ ref('stg_cafeteria_inspections') }}
),

fact_cafeteria_inspection AS (
   SELECT
       {{ dbt_utils.generate_surrogate_key(['record_id']) }} AS inspection_sk,
       s.record_id,

       d1.date_sk AS inspection_date_sk,
       d2.date_sk AS last_inspection_date_sk,

       sch.school_sk,
       v.violation_sk,
       cig.ci_geo_sk,
       cigov.ci_gov_sk,
       sg.geo_sk,

       s.latitude,
       s.longitude,
       CURRENT_TIMESTAMP() AS ci_dw_inserted_at,
   FROM source s
   LEFT JOIN {{ ref('dim_shared_date') }} d1 ON d1.full_date = s.inspection_date
   LEFT JOIN {{ ref('dim_shared_date') }} d2 ON d2.full_date = s.last_inspection_date
   LEFT JOIN {{ ref('dim_school') }} sch ON sch.school_name = s.school_name
   LEFT JOIN {{ ref('dim_violation') }} v ON v.level = s.violation_level AND v.code = s.violation_code
   LEFT JOIN {{ ref('dim_ci_geography') }} cig ON cig.number = s.building_number AND cig.street = s.street_name AND cig.ci_city = s.city AND cig.state = s.state AND cig.bin = s.bin AND cig.ci_bbl = s.bbl AND cig.nta = s.nta AND cig.borocode = s.borocode
   LEFT JOIN {{ ref('dim_ci_government') }} cigov ON cigov.community_board = s.community_board AND cigov.census_tract = s.census_tract
   LEFT JOIN {{ ref('dim_shared_geography') }} sg ON sg.borough = s.borough AND sg.zip_code = s.zip_code AND sg.council_district = s.council_district
)

SELECT *
FROM fact_cafeteria_inspection